# CorrDiff Ensemble Inference Workflow - Modular Version

This notebook demonstrates an advanced ensemble workflow for CorrDiff using modular Python scripts.
All core functionality has been extracted into reusable modules in the `src/` directory.

## Features
- **Modular Design**: Core functionality organized in separate modules
- **Easy Configuration**: Centralized configuration management
- **Ensemble Generation**: Multiple realizations with different seeds
- **Comprehensive Analysis**: Statistical analysis and visualization
- **NetCDF Output**: PhysicsNeMo-compatible output format

## Workflow Overview
1. Load configuration and setup
2. Initialize data source and models
3. Run ensemble inference
4. Save results and analyze statistics
5. Create visualizations

In [ ]:
! chmod -R 777 /app/outputs/generation/

## Import

In [ ]:
import os
import sys
import torch

# Add source modules to path
src_path = '/app/host/home/younes.abid/git/earth2studio/notebooks/tutorials/cordiff/src'
sys.path.append(src_path)

# Import modular components
from trim_coordinates import trim_coordinates
from config import EnsembleConfig
from data_loader import create_data_source
from ensemble_model import create_ensemble_model
from inference import run_ensemble_inference
from output import save_ensemble_netcdf, verify_output
from stats import compute_and_save_stats
from visualization.orchestrator import plot_analysis

print('✅ All modular components imported successfully')

## Trimm coord for reference (should be done only one time)

In [ ]:
# # Trim WRF coordinates to 432x432 for CorrDiff
# wrf_path = "/app/host/home/younes.abid/git/physicsnemo/data/georefrenced/Space42_CorrDiff/wrf_coord.nc"
# trim_pixels = (7, 8, 7, 8)  # (top, bottom, left, right)
# output_path = "/app/host/home/younes.abid/git/physicsnemo/data/georefrenced/Space42_CorrDiff/trimmed_coordinates_432x432.nc"

# trim_coordinates(wrf_path, trim_pixels, output_path)

## Setup and Configuration

In [ ]:
# Create configuration object
config = EnsembleConfig(variables="Rain_rate_LN_Rain_rate_PT")
config.OUTPUT_VARIABLES = ['Rain_rate_LN','Rain_rate_PT']
config.DATA_FILE = config.BASE_DATA_PATH + 'ERA5_WRF_combined_concatenated_432/' + '2022-06-05_2022-11-22_150.nc' 
config.REGRESSION_CHECKPOINT = config.BASE_CHECKPOINTS_PATH + config.VARIABLES + '/checkpoints_regression/UNet.0.590000.mdlus'
config.DIFFUSION_CHECKPOINT = config.BASE_CHECKPOINTS_PATH + config.VARIABLES + '/checkpoints_diffusion/EDMPrecondSuperResolution.0.220000.mdlus'

config.NUM_ENSEMBLES = 10 
config.NUMBER_OF_STEPS = 18
config.INFERENCE_TIMES = [
            #'2022-07-27T00:00:00', '2022-07-27T01:00:00', '2022-07-27T02:00:00', '2022-07-27T03:00:00',
            # '2022-07-27T04:00:00', '2022-07-27T05:00:00', '2022-07-27T06:00:00', '2022-07-27T07:00:00',
            # '2022-07-27T08:00:00', '2022-07-27T09:00:00', '2022-07-27T10:00:00', '2022-07-27T11:00:00',
            # '2022-07-27T12:00:00', '2022-07-27T13:00:00', '2022-07-27T14:00:00', '2022-07-27T15:00:00',
            # '2022-07-27T16:00:00', '2022-07-27T17:00:00', '2022-07-27T18:00:00', '2022-07-27T19:00:00',
            # '2022-07-27T20:00:00', '2022-07-27T21:00:00', '2022-07-27T22:00:00', '2022-07-27T23:00:00',
            
            # '2022-07-28T00:00:00', '2022-07-28T01:00:00', '2022-07-28T02:00:00', '2022-07-28T03:00:00',
            # '2022-07-28T04:00:00', '2022-07-28T05:00:00', '2022-07-28T06:00:00', '2022-07-28T07:00:00',
            # '2022-07-28T08:00:00', '2022-07-28T09:00:00', '2022-07-28T10:00:00', '2022-07-28T11:00:00',
            # '2022-07-28T12:00:00', '2022-07-28T13:00:00', '2022-07-28T14:00:00', '2022-07-28T15:00:00',
            # '2022-07-28T16:00:00', '2022-07-28T17:00:00', '2022-07-28T18:00:00', '2022-07-28T19:00:00',
            # '2022-07-28T20:00:00', '2022-07-28T21:00:00', '2022-07-28T22:00:00', '2022-07-28T23:00:00',
            
            # '2022-07-29T00:00:00', '2022-07-29T01:00:00', '2022-07-29T02:00:00', '2022-07-29T03:00:00',
            # '2022-07-29T04:00:00', '2022-07-29T05:00:00', '2022-07-29T06:00:00', '2022-07-29T07:00:00',
            # '2022-07-29T08:00:00', '2022-07-29T09:00:00', '2022-07-29T10:00:00', '2022-07-29T11:00:00',
            # '2022-07-29T12:00:00', '2022-07-29T13:00:00', '2022-07-29T14:00:00', '2022-07-29T15:00:00',
            # '2022-07-29T16:00:00', '2022-07-29T17:00:00', '2022-07-29T18:00:00', '2022-07-29T19:00:00',
             '2022-07-29T20:00:00', '2022-07-29T21:00:00', '2022-07-29T22:00:00', '2022-07-29T23:00:00'
        ]

# Print configuration summary
config.print_config()

In [ ]:
import xarray as xr
import pandas as pd

# Open file
ds = xr.open_dataset(config.DATA_FILE)

# Get time dimension
time_coord = ds.time
print(f"Total dates: {len(time_coord)}")
print(f"Start: {time_coord.min().values}")
print(f"End: {time_coord.max().values}")

# Check target dates
targets = ['2022-07-29T20:00:00', '2022-07-29T21:00:00', '2022-07-29T22:00:00', '2022-07-29T23:00:00']
time_values = pd.to_datetime(time_coord.values)

print("\nTarget dates:")
for target in targets:
    target_dt = pd.to_datetime(target)
    found = target_dt in time_values
    print(f"{target}: {'✅ Found' if found else '❌ Missing'}")

ds.close()

In [ ]:
time_coord[1145:1160]

In [ ]:
config.DATA_FILE

## Data Source and Model Setup

In [ ]:
# Create data source
data_source = create_data_source(config)

print('✅ Data source created')
print(f'📊 Available samples: {data_source.total_samples}')
print(f'📐 Input grid shape: {data_source.input_shape}')

In [ ]:
# Create ensemble CorrDiff model
ensemble_model = create_ensemble_model(config, data_source)

## Ensemble Inference

In [ ]:
# Run ensemble inference
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
results = run_ensemble_inference(config, ensemble_model, data_source, device)

print(f'📊 Inference complete: {len(results["predictions"])} time steps with {config.NUM_ENSEMBLES} ensemble members each')

## Save Results and Analysis

In [ ]:
# Save results in NetCDF format
saved_file = save_ensemble_netcdf(config, results, ensemble_model)

# Verify the saved file
verify_output(saved_file)

In [ ]:
# Compute and save ensemble statistics
summary_stats, stats_file = compute_and_save_stats(config, saved_file)

In [ ]:
import pprint
pprint.pprint(summary_stats)

## Visualization and Analysis

In [ ]:
# Create comprehensive visualizations
saved_file = '/app/outputs/generation/Rain_rate_LN_Rain_rate_PT/20260205_132916/ensemble_Rain_rate_LN_Rain_rate_PT_20260205_132916.nc'
plot_analysis(config, saved_file, time_idx=0, show=True)

In [ ]:
# Load and inspect the saved data
import xarray as xr

# Open prediction dataset
pred_ds = xr.open_dataset(saved_file, group='prediction')
pred_ds